# Investigation of MUSIN-G Dataset


In [ ]:
if "google.colab" in sys.modules:  # only on Colab; local runs use the uv environment
    %pip install -q braindecode mne mne-bids skorch torcheeg soundfile captum

In [1]:
from pathlib import Path

import braindecode
import mne
import scipy
import sklearn
import skorch
import torch
import torcheeg
import matplotlib
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

from mne_bids import BIDSPath, find_matching_paths, get_entity_vals, read_raw_bids

ModuleNotFoundError: No module named 'braindecode'

In [ ]:
DATA_PATH = Path("data")  # directory path of the data files for our computations
SEED = 42  # random seed for reproducibility

TRAIN = False # Set to True if you want to train the model, otherwise attempt to load model from .pt files

## Load Data

In [ ]:
SONG_NAMES = {
    1: "Trip to the lonely planet",
    2: "Sail",
    3: "Concept 15",
    4: "Aurore",
    5: "Proof",
    6: "Glider",
    7: "Raag Bihag",
    8: "Albela sajan",
    9: "Mor Bani thanghat Kare",
    10: "Fallin",
    11: "Fly Kicks",
    12: "JB",
}

montage = mne.channels.make_standard_montage("GSN-HydroCel-129")

def LoadMUSIN_G(
    filepath: Path = DATA_PATH / "ds003774-download",
    subject_slice: slice = slice(0,20),
    sfreq: int = 125,
) -> pd.DataFrame:
    """
    Load data from https://openneuro.org/datasets/ds003774/versions/1.0.2

    Returns one row per recording (one song for one participant), with columns:
        subject, session, run, song_num, song_name, sfreq, n_times, duration_s,
        wav_path, bids_path, raw

    Note:
        - for 11 participants, the sampling rate of EEG is 1000Hz, for the remaining 9, 250Hz.
            - resample all to 125 Hz by default
        - the 1st session corresponds to song 1 (the README says each session is a song number).
        - `subject_slice` picks which participants to load.
    """
    subjects = get_entity_vals(filepath, "subject")[subject_slice]
    sessions = get_entity_vals(filepath, "session")
    task = get_entity_vals(filepath, "task")[0]

    rows = []
    for sub in subjects:
        print("Subject: " + sub)
        for ses in sessions:
            print("Session: " + ses)
            paths = find_matching_paths(
                subjects=sub,
                sessions=ses,
                tasks=task,
                extensions=[".set"],
                suffixes="eeg",
                root=filepath,
            )
            song_num = int(ses)
            song_file = filepath / "Code" / "ESongs" / f"{song_num}.esh.wav"

            for path in paths:
                raw = read_raw_bids(path)
                raw.resample(sfreq = sfreq)

                # Provided data names the 'Cz" electrode, "E129"
                raw.rename_channels({'E129': 'Cz'})
                raw.set_montage(montage)

                audio, audio_rate = sf.read(song_file)

                raw.load_data()

                rows.append(
                    {
                        "subject": sub,
                        "session": ses,
                        "run": path.run,
                        "song_num": song_num,
                        "song_name": SONG_NAMES[song_num],
                        "wav_path": song_file,
                        # "audio": audio,
                        "audio_rate": audio_rate,
                        "bids_path": path,
                        "raw": raw,
                    }
                )

    return pd.DataFrame(rows)

In [ ]:
df = LoadMUSIN_G()

In [ ]:
df

In [ ]:
df.memory_usage()

## Preprocess data


In [ ]:
# Snapshot one recording before preprocessing (the steps below modify `raw` in place)
PLOT_ROW = 0
raw_before = df["raw"].iloc[PLOT_ROW].copy()


In [ ]:
# Cz is the reference electrode, the data was already referenced to Cz and is flat
# Exclude it from model inputs
df["raw"].apply(lambda x: x.drop_channels("Cz") if "Cz" in x.ch_names else x)

# Cut the 10s of silence from the raw recording
df["raw"].apply(lambda x: x.crop(tmin=10.0))


In [ ]:
# Filter the data
df["raw"].apply(lambda x: x.filter(0.1, 60))

In [2]:
def plot_preprocessing_comparison(raw_before, raw_after, n_chans=4, window_s=(0, 40), title=None):
    """
    Plot a recording before and after preprocessing.

    Top: a few channels over time, on the original recording's time axis, so the
    cropped silence and the filtering are both visible.
    Bottom: power spectral density of both, averaged over channels.
    """
    # Channels present in both (Cz is flat and dropped by preprocessing); evenly spread across the montage
    common = [ch for ch in raw_after.ch_names if ch in raw_before.ch_names]
    picks = [common[i] for i in np.linspace(0, len(common) - 1, n_chans, dtype=int)]

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
    ax_before = fig.add_subplot(gs[0, 0])
    ax_after = fig.add_subplot(gs[0, 1], sharey=ax_before)
    ax_psd = fig.add_subplot(gs[1, :])

    # Unfiltered data carries large per-channel DC offsets; remove each channel's mean
    # within the window for display so the traces can be stacked on a common scale.
    traces = {}
    for label, raw in [("Before", raw_before), ("After", raw_after)]:
        t = raw.times + raw.first_time - raw_before.first_time  # crop shifts `first_time`
        mask = (t >= window_s[0]) & (t <= window_s[1])
        data = raw.copy().pick(picks).get_data()[:, mask] * 1e6  # volts -> microvolts
        traces[label] = (t[mask], data - data.mean(axis=1, keepdims=True))
    spread = 4 * np.std(traces["After"][1])  # same vertical spacing in both panels

    for ax, label in [(ax_before, "Before"), (ax_after, "After")]:
        t, data = traces[label]
        for k, y in enumerate(data):
            ax.plot(t, y - k * spread, lw=0.6)
        ax.set_yticks([-k * spread for k in range(len(picks))], picks)
        ax.set_xlim(window_s)
        ax.set_xlabel("Time in original recording (s)")
        ax.set_title(label + (" (first 10 s are silence, cropped)" if label == "Before" else ""))
    ax_before.set_ylabel("Amplitude (mean-removed, offset, µV)")

    for raw, label in [(raw_before, "Before"), (raw_after, "After")]:
        psd = raw.compute_psd(fmax=raw.info["sfreq"] / 2, picks=common, verbose=False)
        freqs = psd.freqs
        power = 10 * np.log10(psd.get_data() * 1e12)  # V^2/Hz -> dB re 1 µV^2/Hz
        ax_psd.plot(freqs, power.mean(axis=0), label=label)
        ax_psd.fill_between(freqs, np.percentile(power, 10, axis=0), np.percentile(power, 90, axis=0), alpha=0.15)
    ax_psd.axvline(0.1, color="gray", ls=":", lw=1)
    ax_psd.axvline(60, color="gray", ls=":", lw=1)
    ax_psd.set_xscale("log")
    ax_psd.set_xlim(0.1 / 2, None)
    ax_psd.set_xlabel("Frequency (Hz)")
    ax_psd.set_ylabel("Power (dB re 1 µV²/Hz)")
    ax_psd.set_title("PSD: mean over channels, shaded 10th-90th percentile; dotted lines = 0.1 and 60 Hz filter edges")
    ax_psd.legend()
    ax_psd.grid(True, alpha=0.3)

    fig.suptitle(title)
    fig.tight_layout()
    return fig


row = df.iloc[PLOT_ROW]
plot_preprocessing_comparison(
    raw_before,
    df["raw"].iloc[PLOT_ROW],
    title=f"Preprocessing: subject {row['subject']}, \"{row['song_name']}\"",
)
plt.show()


NameError: name 'df' is not defined

## Classify the song based on EEG

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

epoch, song, subject = [], [], []
for index, row in df.iterrows():
    print("Index:", index)
    ep = mne.make_fixed_length_epochs(
        row["raw"].copy(), duration=2, overlap=0
    )
    epoch.append(ep.get_data())
    song.append([row["song_name"]] * len(ep))
    subject.append([row["subject"]] * len(ep))

epoch = np.concatenate(epoch)
song = np.concatenate(song)
subject = np.concatenate(subject)


In [ ]:
# Train test split
gss = GroupShuffleSplit(n_splits=1, test_size=2, random_state=SEED)
train_idx, test_idx = next(gss.split(epoch, song, groups=subject))

Xtrain, Xtest = epoch[train_idx], epoch[test_idx]
ytrain, ytest = song[train_idx], song[test_idx]
subject_train = subject[train_idx]

# Train validation split
# train_idx, valid_idx = next(gss.split(Xtrain, ytrain, groups=subject_train))
# Xtrain, Xvalid = Xtrain[train_idx], Xtrain[valid_idx]
# ytrain, yvalid = ytrain[train_idx], ytrain[valid_idx]

## Train the Model

In [ ]:
from braindecode.models import EEGNet
from braindecode.datautil import infer_signal_properties

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(song)
y_enc = le.transform(song)

ytrain_enc = le.transform(ytrain)
ytest_enc = le.transform(ytest)
# yvalid_enc = le.transform(yvalid)

sig_props = infer_signal_properties(epoch, song, mode="classification")

eegnet_model = EEGNet(
    n_chans=sig_props["n_chans"],
    n_outputs=sig_props["n_outputs"],
    n_times=sig_props["n_times"],
    final_conv_length="auto",
    F1=8,
    D=2,
)



In [ ]:
from braindecode import EEGClassifier
from skorch.helper import predefined_split
from skorch.dataset import ValidSplit, Dataset
from skorch.callbacks import LRScheduler

cuda = torch.cuda.is_available()
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True

max_epochs = 60
batch_size = 64

lr = 0.001
weight_decay = 0

clf = EEGClassifier(
    eegnet_model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    # train_split=predefined_split(Dataset(Xvalid, yvalid_enc)),
    train_split=ValidSplit(0.2, random_state=SEED),
    optimizer__lr=lr,
    optimizer__weight_decay=weight_decay,
    batch_size=batch_size,
    max_epochs=max_epochs,
        callbacks=[
        (
            "lr_scheduler",
            LRScheduler(
                "CosineAnnealingLR", monitor="valid_loss", T_max=max_epochs-1
            ),
        ),
    ],
    device=device,
    classes=[SONG_NAMES[i] for i in range(1, 13)]
)


In [ ]:
if TRAIN:
    clf.fit(Xtrain, ytrain_enc)

    torch.save(clf.state_dict(), "eegnet_classify_song.pt")
else:
    eegnet_model.load_state_dict(torch.load("stimuli_scn_model_csn.pt", weights_only=True))

    eegnet_model.eval()

    clf.initialize()


In [ ]:
from braindecode.visualization import plot_confusion_matrix
from sklearn.metrics import confusion_matrix, accuracy_score

score = accuracy_score(ytest_enc, clf.predict(Xtest))
print("Score:", score)

score = accuracy_score(ytrain_enc, clf.predict(Xtrain))
print("Score:", score)

In [ ]:
from matplotlib import pyplot as plt

def plot_loss_curve(classifiers, labels=None):
    """
    Plot train and validation loss curves for one or more classifiers.

    Parameters
    ----------
    classifiers : list
        List of fitted classifiers with a skorch/braindecode-style `history`.
    labels : list[str], optional
        Names to use for each classifier in the legend.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The generated matplotlib figure.
    """

    if not isinstance(classifiers, (list, tuple)):
        classifiers = [classifiers]

    if labels is None:
        labels = [f"Classifier {i + 1}" for i in range(len(classifiers))]

    if len(labels) != len(classifiers):
        raise ValueError("`labels` must have the same length as `classifiers`.")

    fig, ax = plt.subplots(figsize=(8, 5))

    results_columns = ["train_loss", "valid_loss"]

    for clf, label in zip(classifiers, labels):
        _df = pd.DataFrame(
            clf.history[:, results_columns],
            columns=results_columns,
            index=clf.history[:len(clf.history[:, results_columns]), "epoch"],
        )

        ax.plot(
            _df.index,
            _df["train_loss"],
            linestyle="-",
            label=f"{label} train loss",
        )

        ax.plot(
            _df.index,
            _df["valid_loss"],
            linestyle="--",
            label=f"{label} valid loss",
        )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    return fig

In [ ]:
plot_loss_curve(clf)

In [ ]:
torch.save(eegnet_model.state_dict(), "eegnet.pt")

## Interpretability: DeepLIFT Topographic Maps

DeepLIFT (Shrikumar et al., 2017) attributes the difference between the model's output for an input and its output for a reference (baseline) input to each input feature, i.e. each (channel, time sample). The attributions satisfy *summation-to-delta*: they sum to `f(x) - f(baseline)` for the target class.

- **Baseline:** all-zeros. The data are band-pass filtered (0.1-60 Hz), so zero is the signal mean, a neutral "no activity" reference.
- **Target:** the true song label of each epoch.
- **Topography:** collapse the time axis by taking the mean absolute attribution per channel, average over epochs, and draw it on the GSN-HydroCel-129 montage.


In [ ]:
from captum.attr import DeepLift

# Reload the saved weights so this section runs without retraining
dl_model = EEGNet(
    n_chans=sig_props["n_chans"],
    n_outputs=sig_props["n_outputs"],
    n_times=sig_props["n_times"],
    final_conv_length="auto",
    F1=8,
    D=2,
)
dl_model.load_state_dict(torch.load("eegnet.pt", map_location=device))
dl_model.to(device).eval()  # eval: BatchNorm uses running stats, Dropout disabled


def deeplift_attributions(model, X, y, batch_size=256):
    """
    DeepLIFT attributions for each epoch with respect to its target class.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model in eval mode.
    X : np.ndarray, shape (n_epochs, n_chans, n_times)
    y : np.ndarray, shape (n_epochs,)
        Encoded target class for each epoch.

    Returns
    -------
    attributions : np.ndarray, shape (n_epochs, n_chans, n_times)
    deltas : np.ndarray, shape (n_epochs,)
        Convergence deltas; should be close to zero.
    """
    dl = DeepLift(model)
    attributions, deltas = [], []
    for start in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
        yb = torch.as_tensor(y[start:start + batch_size], dtype=torch.long, device=device)
        attr, delta = dl.attribute(
            xb, baselines=torch.zeros_like(xb), target=yb, return_convergence_delta=True
        )
        attributions.append(attr.detach().cpu().numpy())
        deltas.append(delta.detach().cpu().numpy())
    return np.concatenate(attributions), np.concatenate(deltas)


attr_test, delta_test = deeplift_attributions(dl_model, Xtest, ytest_enc)
print("Test Attributions:", attr_test.shape)
print("Test Max |convergence delta|:", np.abs(delta_test).max())


attr_train, delta_train = deeplift_attributions(dl_model, Xtrain, ytrain_enc)
print("Train Attributions:", attr_train.shape)
print("Train Max |convergence delta|:", np.abs(delta_train).max())


In [ ]:
# Channel order of the epochs matches the raw objects (Cz dropped), so reuse their info
eeg_info = df["raw"].iloc[0].info

# Channel importance: mean |attribution| over time, per epoch
channel_importance = np.abs(attr_test).mean(axis=2)  # (n_epochs, n_chans)


def plot_attribution_topomap(values, info, ax, title):
    im, _ = mne.viz.plot_topomap(values, info, axes=ax, cmap="Reds", show=False)
    ax.set_title(title, fontsize=9)
    return im


# Overall map across all test epochs
fig, ax = plt.subplots(figsize=(5, 5))
im = plot_attribution_topomap(
    channel_importance.mean(axis=0), eeg_info, ax, "DeepLIFT: all test epochs"
)
fig.colorbar(im, ax=ax, shrink=0.7, label="mean |attribution|")
plt.show()


In [ ]:
# Channel order of the epochs matches the raw objects (Cz dropped), so reuse their info
eeg_info = df["raw"].iloc[0].info

# Channel importance: mean |attribution| over time, per epoch
channel_importance = np.abs(attr_train).mean(axis=2)  # (n_epochs, n_chans)

# Overall map across all test epochs
fig, ax = plt.subplots(figsize=(5, 5))
im = plot_attribution_topomap(
    channel_importance.mean(axis=0), eeg_info, ax, "DeepLIFT: all test epochs"
)
fig.colorbar(im, ax=ax, shrink=0.7, label="mean |attribution|")
plt.show()


In [ ]:
# One map per song, on a shared color scale so classes are comparable
class_maps = np.stack(
    [channel_importance[ytest_enc == k].mean(axis=0) for k in range(len(le.classes_))]
)

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
vmax = class_maps.max()
for k, ax in enumerate(axes.flat):
    im, _ = mne.viz.plot_topomap(
        class_maps[k], eeg_info, axes=ax, cmap="Reds", vlim=(0, vmax), show=False
    )
    ax.set_title(le.classes_[k], fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.6, label="mean |attribution|")
fig.suptitle("DeepLIFT channel importance per song (test set)")
plt.show()


In [ ]:
# Same, restricted to epochs the model classified correctly
with torch.no_grad():
    pred_test = np.concatenate([
        dl_model(torch.as_tensor(Xtest[i:i + 256], dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
        for i in range(0, len(Xtest), 256)
    ])
correct = pred_test == ytest_enc
print(f"Correct test epochs: {correct.sum()} / {len(correct)}")

fig, ax = plt.subplots(figsize=(5, 5))
im = plot_attribution_topomap(
    channel_importance[correct].mean(axis=0), eeg_info, ax, "DeepLIFT: correctly classified"
)
fig.colorbar(im, ax=ax, shrink=0.7, label="mean |attribution|")
plt.show()
